# 04 K-Means 聚类分析

基于用户行为特征进行无监督聚类，发现自然用户群体。
- 特征选择与标准化
- 肘部法则确定K值
- 轮廓系数评估聚类质量
- PCA 降维可视化
- 聚类画像与业务命名

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

RANDOM_STATE = 42
from pathlib import Path
BASE_DIR = Path('..').resolve()

## 4.1 加载数据与特征选择

In [ ]:
# 加载用户特征数据
data_path = BASE_DIR / 'data' / 'processed' / 'user_features.parquet'
df = pd.read_parquet(data_path)
print(f"数据加载完成，共 {len(df)} 位用户")

# 选择聚类特征
cluster_features = ['recency_days', 'total_orders', 'total_spend', 'avg_review_score', 'delayed_order_ratio']
available = [f for f in cluster_features if f in df.columns]
missing = [f for f in cluster_features if f not in df.columns]

if missing:
    print(f"⚠️ 以下特征不存在: {missing}")

X = df[available].copy()
for col in X.columns:
    X[col] = X[col].fillna(X[col].median())

print(f"\n已选择 {len(available)} 个特征: {available}")
print(f"数据形状: {X.shape}")
X.describe()

## 4.2 特征标准化

使用 StandardScaler 消除量纲差异，使各特征对聚类贡献均等。

In [ ]:
# StandardScaler 标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"标准化完成，形状: {X_scaled.shape}")
print(f"标准化后均值: {X_scaled.mean(axis=0).round(4)}")
print(f"标准化后标准差: {X_scaled.std(axis=0).round(4)}")

## 4.3 肘部法则 (Elbow Method)

通过 Inertia（簇内平方和）随 K 变化的曲线，寻找"拐点"确定最优聚类数。

In [ ]:
# 肘部法则
k_range = range(2, 9)
inertias = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    print(f"K={k}: Inertia={kmeans.inertia_:.2f}")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=8)
ax.set_xlabel('聚类数 K', fontsize=12)
ax.set_ylabel('Inertia (簇内平方和)', fontsize=12)
ax.set_title('肘部法则 - 确定最优聚类数', fontsize=14)
ax.set_xticks(list(k_range))
ax.grid(True, alpha=0.3)

for k, inertia in zip(k_range, inertias):
    ax.annotate(f'{inertia:.0f}', (k, inertia),
                textcoords="offset points", xytext=(0, 10), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 4.4 轮廓系数 (Silhouette Score)

轮廓系数衡量聚类质量：簇内紧凑度 vs 簇间分离度。越接近 1 越好。

In [ ]:
# 轮廓系数分析
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f"K={k}: 轮廓系数={score:.4f}")

best_k = list(k_range)[np.argmax(silhouette_scores)]
best_score = max(silhouette_scores)
print(f"\n✅ 最优 K = {best_k} (轮廓系数 = {best_score:.4f})")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(list(k_range), silhouette_scores, 'rs-', linewidth=2, markersize=8)
ax.axvline(x=best_k, color='green', linestyle='--', linewidth=1.5, label=f'最优 K={best_k}')
ax.set_xlabel('聚类数 K', fontsize=12)
ax.set_ylabel('轮廓系数', fontsize=12)
ax.set_title('轮廓系数 - 聚类质量评估', fontsize=14)
ax.set_xticks(list(k_range))
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4.5 最终聚类 & PCA 可视化

In [ ]:
# 使用最优 K 执行最终聚类
kmeans_final = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
labels = kmeans_final.fit_predict(X_scaled)
final_score = silhouette_score(X_scaled, labels)

print(f"最终聚类 K={best_k}, 轮廓系数={final_score:.4f}")
print(f"各簇样本量: {pd.Series(labels).value_counts().sort_index().to_dict()}")

# PCA 降维到 2D 可视化
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
explained_var = pca.explained_variance_ratio_

print(f"\nPCA 解释方差: PC1={explained_var[0]:.2%}, PC2={explained_var[1]:.2%}, 累计={sum(explained_var):.2%}")

fig, ax = plt.subplots(figsize=(12, 8))
unique_labels = np.unique(labels)
colors = plt.cm.Set1(np.linspace(0, 1, len(unique_labels)))

for label, color in zip(unique_labels, colors):
    mask = labels == label
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=[color],
               label=f'簇 {label} (n={mask.sum()})', alpha=0.6, s=30,
               edgecolors='white', linewidth=0.5)

ax.set_xlabel(f'PC1 ({explained_var[0]:.1%})', fontsize=12)
ax.set_ylabel(f'PC2 ({explained_var[1]:.1%})', fontsize=12)
ax.set_title('K-Means 聚类结果 - PCA 2D 可视化', fontsize=14)
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4.6 聚类画像分析

In [ ]:
# 各簇特征均值分析
X_with_labels = X.copy()
X_with_labels['cluster'] = labels
cluster_means = X_with_labels.groupby('cluster').mean()

print("各簇特征均值:")
print("=" * 60)
print(cluster_means.round(2).to_string())
print("=" * 60)

# 热力图
fig, ax = plt.subplots(figsize=(12, 6))
cluster_means_scaled = (cluster_means - cluster_means.mean()) / cluster_means.std()

sns.heatmap(
    cluster_means_scaled.T, annot=cluster_means.T.round(2).values,
    fmt='', cmap='RdYlGn', center=0, ax=ax,
    xticklabels=[f'簇 {i}' for i in cluster_means.index],
    yticklabels=cluster_means.columns, linewidths=0.5
)
ax.set_title('各簇特征对比热力图\n(颜色=标准化值, 数字=原始均值)', fontsize=13)
plt.tight_layout()
plt.show()

## 4.7 聚类与 RFM 交叉验证

In [ ]:
# 聚类结果与 RFM 分层交叉分析
segment_col = None
for col in ['rfm_segment', 'rfm_label', 'customer_segment']:
    if col in df.columns:
        segment_col = col
        break

if segment_col:
    cross_df = pd.crosstab(labels, df[segment_col], margins=True, margins_name='合计')
    print("聚类 × RFM 交叉表:")
    print(cross_df.to_string())
    
    # 可视化
    cross_df_no_margin = cross_df.iloc[:-1, :-1]
    fig, ax = plt.subplots(figsize=(12, 6))
    cross_df_no_margin.plot(kind='bar', ax=ax, colormap='Set3')
    ax.set_title('K-Means 聚类 × RFM 分层 交叉分布', fontsize=13)
    ax.set_xlabel('聚类簇')
    ax.set_ylabel('用户数')
    ax.legend(title='RFM 分层', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.tick_params(axis='x', rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("未找到 RFM 分层列，跳过交叉分析")

## 小结

- 通过肘部法则和轮廓系数确定了最优聚类数
- PCA 可视化展示了各簇在特征空间中的分布
- 各簇在消费行为特征上呈现明显差异
- 无监督聚类结果与 RFM 规则分层具有一定的一致性，验证了分层的合理性